<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/03_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Tools an agent can actually use

New domain: a **customer support agent** that triages tickets against real orders and a refund
policy. Different shape from the research agent — narrow, factual, and consequential when wrong.

The lesson underneath: the model never sees your code. It sees a **name, a description, and a
schema**. Those three things are the entire interface, which makes a tool's docstring one of
the highest-leverage prompts in your codebase.

**New in this lesson:** `@tool`, docstrings as prompt surface, error messages as instructions,
and return-shape discipline.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-03-tools"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. The data

Small on purpose. You should be able to read all of it and judge the agent's answers yourself —
if you cannot tell whether the agent is right, you cannot tell whether your tools are working.

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

In [ ]:
# Look at what you are working with.
for order in ORDERS[:3]:
    print(order)
print()
print(TICKETS[0])
print()
print(REFUND_POLICY.strip()[:200], "...")

---

## 2. Three tools, written carefully

Read the docstrings below as if you were the model. They are all you would have.

In [ ]:
# --- snippet:support_agent v1 ---
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID.

    Returns the customer email, item, delivery status, days since order, and price.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    # An error message is an instruction to a reader who cannot see your code.
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits (e.g. 1042). "
        f"Ask the customer to re-check the number on their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


@tool
def get_refund_policy() -> str:
    """Return the full refund policy. Consult this before promising any refund."""
    return REFUND_POLICY
# --- /snippet ---

print("3 tools defined")

In [ ]:
from deepagents import create_deep_agent

SUPPORT_PROMPT = (
    "You are a customer support agent for an office furniture retailer.\n"
    "Always look up the order before answering questions about it.\n"
    "Always check the refund policy before promising a refund, replacement, or exchange.\n"
    "Be concise and specific. Quote the relevant policy line."
)

support_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
)

answer = support_agent.invoke({"messages": [{"role": "user", "content":
    "Ticket T-1: the customer says order 1042 arrived with a cracked leg. What are their options?"
}]})

print(answer["messages"][-1].text)

---

## 3. What the model actually received

Your Python is invisible. This is the entire contract.

In [ ]:
import json

for t in [lookup_order, search_tickets, get_refund_policy]:
    print(f"name:        {t.name}")
    print(f"description: {t.description}")
    print(f"schema:      {json.dumps(t.args_schema.model_json_schema().get('properties', {}))}")
    print("-" * 70)

Notice that the description **is** the docstring, and the schema **is** the type hints. You did
not write a prompt for these tools — but you wrote a prompt for these tools.

This is why "just add a tool" so often fails: the tool works fine, and the model has no idea
when to reach for it.

---

## 4. The same capability, described badly

Identical logic. Different name, no useful description.

In [ ]:
@tool
def get_data(x: str) -> str:
    """Gets data."""
    for order in ORDERS:
        if order["id"] == x:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    raise ValueError("not found")


@tool
def check(q: str) -> str:
    """Checks things."""
    return REFUND_POLICY


bad_agent = create_deep_agent(
    model=MODEL,
    tools=[get_data, check],
    system_prompt=SUPPORT_PROMPT,
)

bad = bad_agent.invoke({"messages": [{"role": "user", "content":
    "Ticket T-1: the customer says order 1042 arrived with a cracked leg. What are their options?"
}]})

print(bad["messages"][-1].text)

In [ ]:
# Count the tool calls each agent made.
def tool_calls(result):
    calls = []
    for m in result["messages"]:
        for tc in getattr(m, "tool_calls", []) or []:
            calls.append(tc["name"])
    return calls

print("well-described tools:", tool_calls(answer))
print("badly-described:     ", tool_calls(bad))

### 🧠 Checkpoint

Open both runs in LangSmith (project `lcw-03-tools`) and compare them.

Which tool did the second agent under-use or skip, and what specifically in the description
caused that?

<details><summary>Show answer</summary>

The second agent typically skips `check` — the refund policy — and answers from the model's
general knowledge of how returns usually work.

The cause is the docstring `Checks things.` The model has to guess whether "checks things" is relevant
to "what are this customer's options", and it frequently guesses no. `get_refund_policy` with
*"Return the full refund policy. Consult this before promising any refund"* removes the guess:
it names the trigger condition.

A useful description answers **"when should I call this?"**, not "what does it do?". The
failure is silent — you get a fluent, confident, unsourced answer — which is exactly the kind
of bug that only shows up in a trace or an eval.

</details>

---

## 5. Error messages are instructions

When a tool fails, its error text goes straight into the model's context. It is a message to a
reader who cannot see your code, cannot read your stack trace, and has to decide what to do
next.

In [ ]:
@tool
def lookup_order_raises(order_id: str) -> str:
    """Look up an order by ID."""
    for order in ORDERS:
        if order["id"] == order_id:
            return str(order)
    raise ValueError(f"KeyError: {order_id}")


@tool
def lookup_order_guides(order_id: str) -> str:
    """Look up an order by ID."""
    for order in ORDERS:
        if order["id"] == order_id:
            return str(order)
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits starting with 10 "
        f"(e.g. 1042). Ask the customer to check their confirmation email, or use "
        f"search_tickets to find the order from the ticket text."
    )


BAD_ID = {"messages": [{"role": "user", "content":
    "Ticket T-8 mentions order 9999 never arrived. Look it up and tell me what to do next."
}]}

for name, t in [("raises", lookup_order_raises), ("guides", lookup_order_guides)]:
    a = create_deep_agent(model=MODEL, tools=[t, search_tickets], system_prompt=SUPPORT_PROMPT)
    print(f"===== {name} =====")
    print(a.invoke(BAD_ID)["messages"][-1].text[:400])
    print()

The guided version usually recovers — it re-reads the ticket, or asks a sensible question. The
raising version tends to apologise and stop.

**Write error strings for the model, the way you would write them for a new colleague.**

---

## 6. Return-shape discipline

A tool that returns everything is a tool that costs you on every future step (lesson 02). The
fix is the same one the harness uses internally: **write the bulk to a file, return the path.**

In [ ]:
from langchain_core.messages.utils import count_tokens_approximately

# Simulate a chatty tool: full ticket history for an order.
def full_history(order_id):
    return "\n".join(
        f"[2026-0{i}-12] agent: standard reply about order {order_id} ... " + ("detail " * 60)
        for i in range(1, 9)
    )

fat = full_history("1042")
print(f"returning everything: ~{count_tokens_approximately([{'role': 'user', 'content': fat}]):,} tokens, every step after this")


@tool
def get_ticket_history(order_id: str) -> str:
    """Fetch the full support history for an order.

    Writes the history to a file and returns the path plus a short summary. Read the file
    only if you need the detail.
    """
    body = full_history(order_id)
    path = f"/tickets/{order_id}.md"
    # In a real tool you would write through the backend; the point here is the return value.
    return (
        f"Wrote {path} ({len(body)} chars, 8 messages). "
        f"Most recent: customer following up on a cracked desk leg. "
        f"Use read_file('{path}') for the full text."
    )


print(f"returning a path:     ~{count_tokens_approximately([{'role': 'user', 'content': get_ticket_history.invoke({'order_id': '1042'})}]):,} tokens")

### 🧠 Checkpoint

Why is "write the blob to a file and return the path" better than simply truncating the blob to
the first 2,000 characters?

<details><summary>Show answer</summary>

Truncation **destroys** information; offloading **defers** it.

With truncation, if the answer was in character 5,000, no amount of subsequent reasoning can
recover it — and neither the model nor you can tell that it was lost. With a file, the model
gets a cheap summary now and can pay for the detail later, on purpose, only when it needs it.

There is a second, subtler benefit: the file persists past the step. Another subagent, a later
turn, or a human debugging the run can all open it. A truncated tool result is gone the moment
the context is compacted.

</details>

### ✍️ Exercise

Below is a broken tool. It has three separate problems:

1. a name that means nothing to a reader who cannot see the code
2. a docstring that does not say when to use it
3. an exception on missing input instead of a usable message

Fix all three, rebuild the agent, and check in the trace that the model now calls it for the
question *"has anyone complained about order 1042 before?"*

```python
@tool
def q(s: str) -> str:
    """Query."""
    hits = [t for t in TICKETS if s.lower() in t["text"].lower()]
    if not hits:
        raise ValueError("nothing")
    return str(hits)
```

<details><summary>Show a solution</summary>

```python
@tool
def search_ticket_text(keyword: str) -> str:
    """Search the text of all past support tickets for a keyword.

    Use this to find out whether a customer has written in before about the same problem,
    or to identify an order when the customer does not know the order number.
    Returns matching ticket IDs with their order IDs and text.
    """
    hits = [t for t in TICKETS if keyword.lower() in t["text"].lower()]
    if not hits:
        return (
            f"No tickets mention {keyword!r}. Try a shorter or more general keyword "
            f"(for example 'desk' rather than 'cracked desk leg')."
        )
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_ticket_text, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
)

print(agent.invoke({"messages": [{"role": "user", "content":
    "Has anyone complained about order 1042 before?"
}]})["messages"][-1].text)
```

</details>

---

## 📌 Key takeaways

- The model sees a **name, a description, and a schema** — those three things are your entire tool interface.
- A good description answers *when should I call this?*, not *what does it do?*
- Badly described tools fail **silently**: you get a fluent, confident, unsourced answer.
- Error messages are instructions to a reader who cannot see your code. Say what to try next.
- Return the smallest thing that lets the agent decide its next step; write bulk to a file and return the path.
- Truncating destroys information, offloading defers it — prefer deferring.
- You test a tool by reading the **trace**, not by checking its return value in isolation.

---

## ➡️ Next

**[04 · Tools II: MCP servers](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/04_mcp.ipynb)**

You wrote those tools. Next: tools you did **not** write and do not control — connecting an
agent to public MCP servers, and the context-budget and trust problems that come with them.